# Imports & File paths

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import pandas as pd
import seaborn as sns
import sqlite3
import numpy as np
from scipy.stats import rankdata, norm
import statsmodels.api as sm
from statsmodels.discrete.count_model import ZeroInflatedPoisson
from statsmodels.stats.multitest import multipletests
from tqdm.auto import tqdm

import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.simplefilter(action='ignore', category=FutureWarning) # to minimise output verbosity later
warnings.simplefilter('ignore', ConvergenceWarning) 

In [ ]:
DB_PATH = 'human_genome_old.db' 

EXPRESSION_OUTLIERS_PATH = '' 
PROTEIN_OUTLIERS_PATH = ''
SPLICING_OUTLIERS_PATH = ''

EXPRESSION_VARIANTS_PATH = ''
PROTEIN_VARIANTS_PATH = ''
SPLICING_VARIANTS_PATH = ''

SAMPLE_ID = 'sampleID' # change if sample ID column has a different name

In [ ]:
# outliers
outliers_exp = pd.read_parquet(EXPRESSION_OUTLIERS_PATH)
outliers_prot = pd.read_parquet(PROTEIN_OUTLIERS_PATH)
outliers_spl = pd.read_parquet(SPLICING_OUTLIERS_PATH)

# variants
variants_exp = pd.read_parquet(EXPRESSION_VARIANTS_PATH)
variants_prot = pd.read_parquet(PROTEIN_VARIANTS_PATH)
variants_spl = pd.read_parquet(SPLICING_VARIANTS_PATH)

In [ ]:
# get generic gene names instead of symbols, if necessary
conn = sqlite3.connect(DB_PATH)
query = "SELECT ensembl_gene_id, symbol FROM genes"
gene_mapping = pd.read_sql(query, conn)
conn.close()
gene_mapping['ensembl_gene_id'] = gene_mapping['ensembl_gene_id'].str.split('.').str[0].str.strip()
id_to_symbol = gene_mapping.dropna().set_index('ensembl_gene_id')['symbol'].to_dict()

In [ ]:
def map_and_clean_ids(df, id_column):
    df['geneID_filtered'] = df[id_column].astype(str).str.split('.').str[0].str.strip() # just for double security
    df['geneID_filtered'] = df['geneID_filtered'].map(id_to_symbol).fillna(df['geneID_filtered'])
    return df

outliers_exp = map_and_clean_ids(outliers_exp, 'geneID_short') # adjust outlier column if different
outliers_prot = map_and_clean_ids(outliers_prot, 'geneID_short')
outliers_spl = map_and_clean_ids(outliers_spl, 'geneID_short')

variants_exp = map_and_clean_ids(variants_exp, 'gene_name')
variants_prot = map_and_clean_ids(variants_prot, 'gene_name')
variants_spl = map_and_clean_ids(variants_spl, 'gene_name')

# Statistical Tests

In [ ]:
def calculate_burdens(outlier_df, sample_id=SAMPLE_ID):
    # calculate total number of unique gene outliers per sample
    burdens = outlier_df.groupby(sample_id)['geneID_filtered'].nunique().reset_index(name='burden')
    return burdens

In [ ]:
def get_testable_genes(outlier_df, variant_df, min_carriers=5):
    outlier_genes = set(outlier_df['geneID_filtered'].unique())
    res_filtered = variant_df[variant_df['geneID_filtered'].isin(outlier_genes)]

    carrier_counts = (
        res_filtered.drop_duplicates(subset=['geneID_filtered', SAMPLE_ID])
        ['geneID_filtered']
        .value_counts()
    )

    valid_genes = carrier_counts[carrier_counts >= min_carriers].index.tolist()

    print(f"Total testable genes: {len(valid_genes)}")
    return valid_genes

In [ ]:
def stats_test(genes_to_test, variant_df, burden_df, filter_sig=True, alpha=0.05):
    if len(genes_to_test) == 0:
        return pd.DataFrame()

    burden_df = burden_df.copy()
    burden_df['global_rank'] = rankdata(burden_df['burden'])
    N = len(burden_df)

    res_filtered = variant_df[variant_df['geneID_filtered'].isin(genes_to_test)]
    unique_carriers = res_filtered.drop_duplicates(subset=['geneID_filtered', SAMPLE_ID])

    merged = unique_carriers.merge(
        burden_df[[SAMPLE_ID, 'global_rank']],
        on=SAMPLE_ID,
        how='inner'
    )

    stats_df = merged.groupby('geneID_filtered')['global_rank'].agg(
        n_carriers='count',
        rank_sum='sum'
    ).reset_index()
    stats_df.rename(columns={'geneID_filtered': 'Gene'}, inplace=True)

    n1 = stats_df['n_carriers']
    n2 = N - n1
    s = stats_df['rank_sum']

    expected = n1 * (N + 1) / 2.0
    std_dev = np.sqrt(n1 * n2 * (N + 1) / 12.0)

    z_scores = (s - expected) / std_dev
    stats_df['p_val'] = norm.sf(z_scores)

    stats_df = stats_df.drop(columns=['rank_sum'])

    if not stats_df.empty:
        _, stats_df['padj'], _, _ = multipletests(stats_df['p_val'], method='fdr_bh')

        if filter_sig:
            return stats_df[stats_df['padj'] < alpha].sort_values('padj')
        else:
            return stats_df.sort_values('p_val')

    return stats_df

In [ ]:
# gene expression
burden_exp = calculate_burdens(outrider)
genes_exp = get_testable_genes(outrider, variants_exp, min_carriers=20)
sig_exp = stats_test(genes_exp, variants_exp, burden_exp, filter_sig=True, alpha=0.05)

# protein
burden_prot = calculate_burdens(protein)
genes_prot = get_testable_genes(protein, variants_prot, min_carriers=20)
sig_prot = stats_test(genes_prot, variants_prot, burden_prot, filter_sig=True, alpha=0.1)

# splicing
burden_spl = calculate_burdens(splicing)
genes_spl = get_testable_genes(splicing, variants_spl, min_carriers=20)
sig_spl = stats_test(genes_spl, variants_spl, burden_spl, filter_sig=True, alpha=0.05)

In [ ]:
def plot_sig_gene_results(gene_list, variant_df, burden_df, stats_df, modality_name="Variant"):
    for gene in gene_list:
        gene_stats_subset = stats_df[stats_df['Gene'] == gene]
        if gene_stats_subset.empty:
            print(f"Skipping {gene}: Not found in stats_df")
            continue
            
        gene_stats = gene_stats_subset.iloc[0]
        p_val = gene_stats['p_val']
        p_adj = gene_stats['padj']

        gene_data = variant_df[variant_df['geneID_filtered'] == gene].copy()

        def sum_small_variant(group):
            muts = set(group['ref'].astype(str) + ">" + group['alt'].astype(str))
            return "/".join(sorted(muts))

        if not gene_data.empty:
            variant_summary = gene_data.groupby(SAMPLE_ID).apply(sum_small_variant).reset_index(name='Variant_Detail')
        else:
            variant_summary = pd.DataFrame(columns=[SAMPLE_ID, 'Variant_Detail'])

        df_plot = pd.merge(burden_df, variant_summary, on=SAMPLE_ID, how='left')
        
        df_plot['Group'] = np.where(df_plot['Variant_Detail'].isna(), 'No Variant', f'Has {modality_name}')
        df_plot['Variant_Detail'] = df_plot['Variant_Detail'].fillna('No Variant')

        plt.figure(figsize=(10, 7))

        sns.boxplot(data=df_plot, x='Group', y='burden',
                    palette={'No Variant': '#f5f5f5', f'Has {modality_name}': 'white'},
                    order=['No Variant', f'Has {modality_name}'], showfliers=False, width=0.5)

        unique_variants = [v for v in df_plot['Variant_Detail'].unique() if v != 'No Variant']
        
        if len(unique_variants) > 10:
            hue_col = 'Group'
            palette_dict = {'WT': '#cccccc', f'Has {modality_name}': '#e78ac3'}
        else:
            hue_col = 'Variant_Detail'
            colors = sns.color_palette("Set2", len(unique_variants))
            palette_dict = dict(zip(unique_variants, colors))
            palette_dict['No Variant'] = '#cccccc'

        sns.stripplot(data=df_plot, x='Group', y='burden',
                      hue=hue_col,
                      palette=palette_dict,
                      order=['WT', f'Has {modality_name}'],
                      jitter=True, s=8, alpha=0.7)

        plt.title(f"{gene}\n(p: {p_val:.2e}, p adj: {p_adj:.2e})", fontsize=15, pad=20)
        plt.ylabel("Total Outlier Burden", fontsize=12)
        plt.xlabel("")

        handles, labels = plt.gca().get_legend_handles_labels()
        filtered_indices = [i for i, label in enumerate(labels) if label not in ['No Variant', 'WT']]
        
        if filtered_indices:
            plt.legend([handles[i] for i in filtered_indices],
                       [labels[i] for i in filtered_indices],
                       title="Mutation (Ref>Alt)" if hue_col == 'Variant_Detail' else "Variant",
                       bbox_to_anchor=(1.05, 1), loc='upper left')
        else:
            plt.legend([],[], frameon=False) # Hide legend if empty

        n_wt = len(df_plot[df_plot['Group'] == 'WT'])
        n_carrier = len(df_plot[df_plot['Group'] == f'Has {modality_name}'])
        plt.xticks([0, 1], [f'WT\n(n={n_wt})', f'Has {modality_name}\n(n={n_carrier})'])

        plt.tight_layout()
        
        filename_modality = modality_name.lower().replace(" ", "_")
        plt.savefig(f'{filename_modality}_outliers_{gene}.png', dpi=300, bbox_inches='tight')
        plt.show()

In [ ]:
# eg for gene exp
top_n_exp = sig_exp.head(5)['Gene'].tolist()
plot_sig_gene_results(top_n_exp, variants_exp, burden_exp, sig_exp, modality_name="Expression Variant")